In [ ]:
!pip install pandas openpyxl requests --quiet

In [ ]:
# Imports and configuration
import re
import json
import urllib.request
import unicodedata
import pandas as pd
from collections import Counter
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

INPUT_FILE  = 'FTData.xlsx'
OUTPUT_FILE = 'Q2_Cleanup_Results.xlsx'

print('Configuration loaded.')
print(f'  Input  : {INPUT_FILE}')
print(f'  Output : {OUTPUT_FILE}')

Configuration loaded.
  Input  : FTData.xlsx
  Output : Q2_Cleanup_Results.xlsx


In [ ]:
# Fetch transcriptions from GCP


def fetch_transcription(recording_id, url_user_id, timeout=10):
    """
    Fetch transcription JSON using the upload_goai URL format
    as described in the task instructions.
    Returns (full_text_string, raw_segments_list)
    """
    url = f'https://storage.googleapis.com/upload_goai/{url_user_id}/{recording_id}_transcription.json'
    try:
        with urllib.request.urlopen(url, timeout=timeout) as r:
            data = json.loads(r.read())
        if isinstance(data, list):
            texts = [seg.get('text', '') for seg in data if isinstance(seg, dict)]
            return ' '.join(t for t in texts if t), data
        return str(data), []
    except Exception:
        return None, None


df_raw = pd.read_excel(INPUT_FILE)
transcriptions = []
print('Fetching transcriptions...')

for _, row in df_raw.iterrows():
    # Extract user_id from URL path
    m = re.search(r'/hi/(\d+)/(\d+)_', row['transcription_url_gcp'])
    if not m:
        continue
    url_user_id = m.group(1)
    text, raw = fetch_transcription(row['recording_id'], url_user_id)
    if text:
        transcriptions.append({
            'recording_id': row['recording_id'],
            'text': text,
            'raw': raw
        })
        print(f'  ✓ {row["recording_id"]} ({len(text)} chars, {len(raw)} segments)')
    else:
        print(f'  ✗ {row["recording_id"]} — fetch failed')

print(f'\nTotal: {len(transcriptions)} recordings fetched.')
print()
# Preview first segment
if transcriptions:
    print('Sample text (first recording, first 400 chars):')
    print(transcriptions[0]['text'][:400])

Fetching transcriptions...
  ✓ 825780 (2984 chars, 24 segments)
  ✓ 825727 (3167 chars, 38 segments)
  ✓ 988596 (2674 chars, 29 segments)
  ✓ 990175 (3193 chars, 25 segments)
  ✓ 526266 (3087 chars, 40 segments)
  ✓ 520199 (2038 chars, 47 segments)
  ✓ 542785 (3407 chars, 58 segments)
  ✓ 494019 (3536 chars, 48 segments)
  ✓ 523045 (4496 chars, 43 segments)
  ✓ 522951 (2910 chars, 38 segments)
  ✓ 254219 (4901 chars, 52 segments)
  ✓ 253253 (4278 chars, 61 segments)
  ✓ 351501 (3496 chars, 49 segments)
  ✓ 350606 (2989 chars, 43 segments)
  ✓ 629904 (2475 chars, 46 segments)
  ✓ 635909 (4225 chars, 50 segments)
  ✓ 989901 (6802 chars, 106 segments)
  ✓ 990783 (6941 chars, 120 segments)
  ✓ 240907 (3252 chars, 38 segments)
  ✓ 240909 (2657 chars, 26 segments)
  ✓ 270153 (4045 chars, 48 segments)
  ✓ 270150 (3471 chars, 56 segments)
  ✓ 475392 (2187 chars, 41 segments)
  ✓ 475356 (1202 chars, 38 segments)
  ✓ 255349 (5104 chars, 53 segments)
  ✓ 255381 (5123 chars, 57 segments)
  ✓ 76768

In [ ]:

# ==========================================================================
# Strategy:
#   1. Build lookup tables for all Hindi number words
#   2. Tokenise text and scan left-to-right
#   3. When a number word is found:
#      - If followed by a LARGE multiplier (सौ/हजार/लाख): chain them (दो सौ → 200)
#      - Otherwise: convert only that single word (दो तीन → 2 3, not 5)
#   4. Skip ORDINALS (पहला, दूसरा) — context-dependent, conservative
#   5. Skip IDIOMS (दो-चार, एक से एक) — not numeric
# ==========================================================================

# ── Lookup tables ──────────────────────────────────────────────────────────
UNITS = {
    'शून्य': 0, 'एक': 1, 'दो': 2, 'तीन': 3, 'चार': 4,
    'पाँच': 5, 'पांच': 5, 'छह': 6, 'छः': 6, 'छै': 6,
    'सात': 7, 'आठ': 8, 'नौ': 9,
}
TEENS = {
    'दस': 10, 'ग्यारह': 11, 'बारह': 12, 'तेरह': 13, 'चौदह': 14,
    'पंद्रह': 15, 'सोलह': 16, 'सत्रह': 17, 'अठारह': 18, 'उन्नीस': 19,
}
TENS = {
    'बीस': 20, 'तीस': 30, 'चालीस': 40, 'पचास': 50,
    'साठ': 60, 'सत्तर': 70, 'अस्सी': 80, 'नब्बे': 90,
}
COMPOUND = {
    'इक्कीस':21,'बाईस':22,'तेईस':23,'चौबीस':24,'पच्चीस':25,
    'छब्बीस':26,'सत्ताईस':27,'अट्ठाईस':28,'उनतीस':29,
    'इकतीस':31,'बत्तीस':32,'तैंतीस':33,'चौंतीस':34,'पैंतीस':35,
    'छत्तीस':36,'सैंतीस':37,'अड़तीस':38,'उनतालीस':39,
    'इकतालीस':41,'बयालीस':42,'तैंतालीस':43,'चौवालीस':44,'पैंतालीस':45,
    'छियालीस':46,'सैंतालीस':47,'अड़तालीस':48,'उनचास':49,
    'इक्यावन':51,'बावन':52,'तिरपन':53,'चौवन':54,'पचपन':55,
    'छप्पन':56,'सत्तावन':57,'अठावन':58,'उनसठ':59,
    'इकसठ':61,'बासठ':62,'तिरसठ':63,'चौसठ':64,'पैंसठ':65,
    'छियासठ':66,'सड़सठ':67,'अड़सठ':68,'उनहत्तर':69,
    'इकहत्तर':71,'बहत्तर':72,'तिहत्तर':73,'चौहत्तर':74,'पचहत्तर':75,
    'छिहत्तर':76,'सतहत्तर':77,'अठहत्तर':78,'उनासी':79,
    'इक्यासी':81,'बयासी':82,'तिरासी':83,'चौरासी':84,'पचासी':85,
    'छियासी':86,'सत्तासी':87,'अठासी':88,'नवासी':89,
    'इक्यानवे':91,'बानवे':92,'तिरानवे':93,'चौरानवे':94,'पचानवे':95,
    'छियानवे':96,'सत्तानवे':97,'अट्ठानवे':98,'निन्यानवे':99,
}
LARGE = {
    'सौ': 100, 'हज़ार': 1000, 'हजार': 1000,
    'लाख': 100000, 'करोड़': 10000000,
}
ENGLISH_NUMS_DEV = {
    'नाइन्टीन': 19, 'नाइन्टीस': 90,
    'थर्ड': 3,   'फोर्ट': 40,
    'ट्वेंटी': 20, 'थर्टी': 30, 'फोर्टी': 40, 'फिफ्टी': 50,
}
ALL_SINGLE = {}
ALL_SINGLE.update(UNITS); ALL_SINGLE.update(TEENS)
ALL_SINGLE.update(TENS);  ALL_SINGLE.update(COMPOUND)
ALL_SINGLE.update(ENGLISH_NUMS_DEV)

ORDINALS = {
    'पहला','पहली','पहले',
    'दूसरा','दूसरी','दूसरे',
    'तीसरा','तीसरी','तीसरे',
    'चौथा','चौथी','चौथे',
}

IDIOM_PATTERNS = [
    r'(?:दो|तीन|चार|पाँच|पांच)-(?:दो|तीन|चार|पाँच|पांच|छह)\s+\w+',
    r'एक\s+से\s+एक',
    r'एक-दो',
]


def is_idiomatic(text, position):
    context = text[max(0, position - 20): position + 50]
    return any(re.search(p, context) for p in IDIOM_PATTERNS)


def parse_number_phrase(tokens):
    """
    Parse one number word (optionally followed by a LARGE multiplier).

    KEY DESIGN DECISION:
    Only chains tokens when a LARGE multiplier (सौ/हजार/लाख) is present.
    Adjacent plain numbers (दो तीन) are converted INDEPENDENTLY:
      दो तीन लोग   → 2 3 लोग   (not 5 लोग)
      दो सौ रुपया  → 200 रुपया  (multiplier present → chain)
      दो हजार बारह → 2012       (multiplier + remainder → chain)

    Returns (integer_value, tokens_consumed)
    """
    if not tokens:
        return None, 0

    tok = tokens[0]

    # Must start with a known number word
    if tok not in ALL_SINGLE and tok not in LARGE:
        return None, 0

    # Standalone large multiplier (e.g., "सौ" alone = 100)
    if tok in LARGE:
        return LARGE[tok], 1

    base     = ALL_SINGLE[tok]
    consumed = 1
    total    = 0
    current  = base
    i        = 1

    # Look ahead: only chain if a LARGE multiplier follows
    while i < len(tokens):
        next_tok = tokens[i]
        if next_tok in LARGE:
            multiplier = LARGE[next_tok]
            if current == 0:
                current = 1
            total    += current * multiplier
            current   = 0
            consumed  = i + 1
            i        += 1
            # Remainder after multiplier (e.g., दो हजार + बारह = 2012)
            if i < len(tokens) and tokens[i] in ALL_SINGLE:
                total   += ALL_SINGLE[tokens[i]]
                consumed = i + 1
                i       += 1
            break
        else:
            # Next word is also a plain number — STOP, don't chain
            break

    total += current
    return total, consumed


def normalize_numbers(text):
    """
    Convert Hindi number words in text to digits.

    Returns:
        normalized_text   : str  — text with number words replaced by digits
        replacements      : list — [(original_phrase, digit_string, position), ...]
    """
    tokens_with_pos = [(m.group(), m.start(), m.end())
                       for m in re.finditer(r'\S+', text)]
    tokens       = [t for t, _, _ in tokens_with_pos]
    result_tokens = list(tokens)
    replacements  = []
    skip_until    = -1
    i             = 0

    while i < len(tokens):
        if i <= skip_until:
            i += 1
            continue

        tok       = tokens[i]
        tok_start = tokens_with_pos[i][1]

        if tok in ORDINALS:
            i += 1
            continue

        if is_idiomatic(text, tok_start):
            i += 1
            continue

        value, consumed = parse_number_phrase(tokens[i:])

        if value is not None and consumed > 0:
            original_phrase = ' '.join(tokens[i: i + consumed])
            replacements.append((original_phrase, str(value), tok_start))
            result_tokens[i] = str(value)
            for j in range(1, consumed):
                result_tokens[i + j] = ''   # mark consumed tokens
            skip_until = i + consumed - 1
            i += consumed
        else:
            i += 1

    normalized = re.sub(r'\s+', ' ',
                        ' '.join(t for t in result_tokens if t != '')).strip()
    return normalized, replacements


# ── Verification ───────────────────────────────────────────────────────────
test_cases = [
    ('दो तीन लोग',              '2 3 लोग'),
    ('छः सात आठ किलोमीटर',    '6 7 8 किलोमीटर'),
    ('दो सौ रुपया',             '200 रुपया'),
    ('दो हजार बारह',            '2012'),
    ('सात आठ मिनट',            '7 8 मिनट'),
    ('पूरे अस्सी घाट',         'पूरे 80 घाट'),
    ('तीस पैंतीस घाट',         '30 35 घाट'),
    ('दस बजे',                  '10 बजे'),
    ('इक्कीस का डेट',           '21 का डेट'),
    ('नाइन्टीन नाइन्टीस के सॉन्ग', '19 90 के सॉन्ग'),
    ('दो-चार बातें',            'दो-चार बातें'),   # idiom — preserve
    ('एक से एक डिजाइन',        'एक से एक डिजाइन'),  # idiom — preserve
    ('पहला बार था',             'पहला बार था'),      # ordinal — preserve
]

print('Number normalizer verification:')
print(f'{"Input":<35} {"Expected":<30} {"Got":<30} {"OK?"}')
print('-' * 105)
all_pass = True
for inp, expected in test_cases:
    result, _ = normalize_numbers(inp)
    ok = result == expected
    if not ok:
        all_pass = False
    flag = '✅' if ok else '❌'
    print(f'{inp:<35} {expected:<30} {result:<30} {flag}')
print()
print('All tests passed!' if all_pass else 'SOME TESTS FAILED — check the code above')

Number normalizer verification:
Input                               Expected                       Got                            OK?
---------------------------------------------------------------------------------------------------------
दो तीन लोग                          2 3 लोग                        2 3 लोग                        ✅
छः सात आठ किलोमीटर                  6 7 8 किलोमीटर                 6 7 8 किलोमीटर                 ✅
दो सौ रुपया                         200 रुपया                      200 रुपया                      ✅
दो हजार बारह                        2012                           2012                           ✅
सात आठ मिनट                         7 8 मिनट                       7 8 मिनट                       ✅
पूरे अस्सी घाट                      पूरे 80 घाट                    पूरे 80 घाट                    ✅
तीस पैंतीस घाट                      30 35 घाट                      30 35 घाट                      ✅
दस बजे                              10 बजे                  

In [ ]:
#  Part a: Edge Case Analysis
# These are real phrases from the dataset where the correct behavior
# requires a judgment call, not just a lookup.

edge_cases = [
    (
        'दो-चार बातें',
        'PRESERVE',
        'Idiomatic: \'a few words\'. Hyphenated numeral pair signals '
        'indefinite/approximate quantity, not a specific count. '
        'Converting → \'2-4 बातें\' is misleading and changes the meaning.'
    ),
    (
        'एक से एक डिजाइन',
        'PRESERVE',
        'Idiomatic: \'outstanding designs one after another\' (superlative). '
        '\'एक से एक\' is a fixed idiom meaning extraordinary. Converting would '
        'produce \'1 से 1 डिजाइन\' which is grammatically broken.'
    ),
    (
        'पहला बार था',
        'PRESERVE',
        'Ordinal: \'it was the first time\'. पहला is an ordinal adjective, '
        'not a cardinal count. Conservative approach: preserve all ordinals '
        'to avoid downstream confusion (NLP models handle पहला correctly).'
    ),
    (
        'एक देखना था',
        'CONVERT → 1 देखना था',
        'Determiner use: \'there was one thing to see\'. एक here functions as '
        'a cardinal determiner. Converting is technically correct and follows '
        'ASR post-processing convention. The meaning is preserved.'
    ),
    (
        'दो तीन लोग',
        'CONVERT → 2 3 लोग',
        'Approximate range: \'two or three people\'. Convert each number '
        'independently. Standard ASR post-processing practice: approximate '
        'ranges are represented as individual digits, not as a sum.'
    ),
    (
        'नौ बजे',
        'CONVERT → 9 बजे',
        'Time expression: \'nine o\'clock\'. Time expressions unambiguously '
        'benefit from digit form for downstream time parsing and NLP tasks.'
    ),
    (
        'दो हजार बारह',
        'CONVERT → 2012',
        'Year: multiplier (हजार) present → chain numbers. '
        'दो × हजार + बारह = 2012. Clear year reference, conversion correct.'
    ),
    (
        'नाइन्टीन नाइन्टीस के सॉन्ग',
        'CONVERT → 19 90 के सॉन्ग',
        'English decade reference: \'songs of the nineties\'. Each English '
        'number word converted independently (no multiplier). '
        '19 90 is the best we can do without decade-specific logic.'
    ),
]

print('EDGE CASES — Judgment Calls')
print('=' * 70)
for phrase, decision, reasoning in edge_cases:
    result, reps = normalize_numbers(phrase)
    print(f'Input    : {phrase}')
    print(f'Output   : {result}')
    print(f'Decision : {decision}')
    print(f'Reason   : {reasoning}')
    print()

EDGE CASES — Judgment Calls
Input    : दो-चार बातें
Output   : दो-चार बातें
Decision : PRESERVE
Reason   : Idiomatic: 'a few words'. Hyphenated numeral pair signals indefinite/approximate quantity, not a specific count. Converting → '2-4 बातें' is misleading and changes the meaning.

Input    : एक से एक डिजाइन
Output   : एक से एक डिजाइन
Decision : PRESERVE
Reason   : Idiomatic: 'outstanding designs one after another' (superlative). 'एक से एक' is a fixed idiom meaning extraordinary. Converting would produce '1 से 1 डिजाइन' which is grammatically broken.

Input    : पहला बार था
Output   : पहला बार था
Decision : PRESERVE
Reason   : Ordinal: 'it was the first time'. पहला is an ordinal adjective, not a cardinal count. Conservative approach: preserve all ordinals to avoid downstream confusion (NLP models handle पहला correctly).

Input    : एक देखना था
Output   : 1 देखना था
Decision : CONVERT → 1 देखना था
Reason   : Determiner use: 'there was one thing to see'. एक here functions as a cardinal

In [ ]:
#  Part b: English Word Detector
# ==========================================================================
# Context English words spoken in Hindi conversation
# are transcribed in Devanagari. We need to IDENTIFY and TAG these words.
# This is for downstream processing — it does NOT mean these are errors.
#
# Strategy (3 layers):
#   1. Roman script → always English
#   2. In known loanword list → English
#   3. Phonotactic patterns typical of English transliteration → English
# ==========================================================================

KNOWN_ENGLISH_DEVANAGARI = {
    # Technology
    'प्रोजेक्ट','नेटवर्क','इंटरनेट','कंप्यूटर','लैपटॉप','मोबाइल',
    'फोन','वीडियो','ऑडियो','कैमरा','चार्जर','स्क्रीन','ऐप',
    # Entertainment
    'सॉन्ग','म्यूजिक','सीरीज','मूवी','डीजे','पॉप','रॉक','क्लासिक',
    'फिल्म',
    # Social / Digital
    'इमोशनल','ट्रोल','इंग्नॉर','शेयर','फॉलो','लाइक','कमेंट',
    # Fashion
    'आउटफिट','स्टाइल','फैशन','ट्रेंड','बोल्ड','जीन्स',
    # Places / Activities
    'होटल','मार्केट','कैंप','कैंपिंग','एडवेंचर','बॉर्डर','स्टेशन',
    # Professional
    'प्रेजेंटेशन','इंटरव्यू','मैनेजर','सीनियर','ऑफिस','मीटिंग',
    # General English words commonly code-switched in Hindi speech
    'एरिया','टॉपिक','एक्सपीरियंस','मेजरमेंट','एटमॉस्फियर',
    'कार्ड','सेटअप','एनर्जी','पॉजिटिव','नेगेटिव','एक्चुअली',
    'सीरियसली','बेसिकली','एग्जेक्टली','प्रॉपर','नॉर्मल','स्पेशल',
    # Address forms (English-origin)
    'मैडम','मैम','हेलो','हेल्लो',
    # Common verbs from English
    'ट्राई','कनेक्ट','डांस','ट्रॉय',
}

# Phonotactic patterns that distinguish English loanwords from Hindi words.
# These are character sequences very rare in native Hindi vocabulary
# but common in English transliterations.
ENGLISH_PHONOTACTIC_PATTERNS = [
    r'\u0911',          # ऑ — short 'o' (boss, office, online) — very rare in Hindi
    r'\u0949',          # ॉ  — same vowel as matra
    r'इंग$',            # -ing suffix
    r'शन$',             # -tion → -शन
    r'मेंट$',           # -ment → -मेंट
    r'नेस$',            # -ness → -नेस
    r'ि?टी$',           # -ity → -िटी
    r'एबल$',            # -able → -एबल
    r'ि?फुल$',          # -ful → -फुल
    r'लेस$',            # -less → -लेस
]


def get_script(word):
    has_dev = any('\u0900' <= c <= '\u097f' for c in word)
    has_lat = any('A' <= c <= 'Z' or 'a' <= c <= 'z' for c in word)
    if has_dev and not has_lat: return 'devanagari'
    if has_lat and not has_dev: return 'latin'
    return 'mixed' if (has_dev and has_lat) else 'other'


def is_english_word(word):
    """
    Returns True if the word is identified as English.

    Priority:
      1. Roman script  → English
      2. Known loanword list → English
      3. Phonotactic pattern → English
      4. Otherwise → Hindi
    """
    w = re.sub(r'[\u0964\u0965,.!?;:\-\u2013\u2014\u2026"\' ()\[\]]+', '', word).strip()
    if not w:
        return False
    if get_script(w) == 'latin':
        return True
    if w in KNOWN_ENGLISH_DEVANAGARI:
        return True
    for pattern in ENGLISH_PHONOTACTIC_PATTERNS:
        if re.search(pattern, w):
            return True
    return False


def tag_english_words(text):
    """
    Wrap English words with [EN]...[/EN] tags.

    Example:
      Input : मेरा इंटरव्यू बहुत अच्छा गया
      Output: मेरा [EN]इंटरव्यू[/EN] बहुत अच्छा गया

    Returns (tagged_text, list_of_english_words_found)
    """
    tokens       = text.split()
    tagged       = []
    english_found = []

    for token in tokens:
        # Preserve surrounding punctuation
        m = re.match(r'^([\u0964\u0965,.!?;:\-]*)(.*?)([\u0964\u0965,.!?;:\-]*)$',
                     token, re.DOTALL)
        prefix, word, suffix = (m.groups() if m else ('', token, ''))

        if word and is_english_word(word):
            tagged.append(f'{prefix}[EN]{word}[/EN]{suffix}')
            english_found.append(word)
        else:
            tagged.append(token)

    return ' '.join(tagged), english_found


# ── Verification ───────────────────────────────────────────────────────────
tag_tests = [
    'मेरा इंटरव्यू बहुत अच्छा गया',
    'ये problem solve नहीं हो रहा',
    'हमारा प्रोजेक्ट भी था उस एरिया में',
    'बोल्ड आउटफिट नहीं पहनना चाहिए',
    'नेटवर्क ही ऐसा है मैं क्या करूं',
    'जंगल में कैंपिंग और एडवेंचर करने गए',
    'पुराने सॉन्ग सुनने में मज़ा आता है',
    'आपका टॉपिक क्या है मैडम',
]

print('English Word Detection Verification:')
print()
for test in tag_tests:
    tagged, en_words = tag_english_words(test)
    print(f'Input : {test}')
    print(f'Output: {tagged}')
    print(f'EN    : {en_words}')
    print()

English Word Detection Verification:

Input : मेरा इंटरव्यू बहुत अच्छा गया
Output: मेरा [EN]इंटरव्यू[/EN] बहुत अच्छा गया
EN    : ['इंटरव्यू']

Input : ये problem solve नहीं हो रहा
Output: ये [EN]problem[/EN] [EN]solve[/EN] नहीं हो रहा
EN    : ['problem', 'solve']

Input : हमारा प्रोजेक्ट भी था उस एरिया में
Output: हमारा [EN]प्रोजेक्ट[/EN] भी था उस [EN]एरिया[/EN] में
EN    : ['प्रोजेक्ट', 'एरिया']

Input : बोल्ड आउटफिट नहीं पहनना चाहिए
Output: [EN]बोल्ड[/EN] [EN]आउटफिट[/EN] नहीं पहनना चाहिए
EN    : ['बोल्ड', 'आउटफिट']

Input : नेटवर्क ही ऐसा है मैं क्या करूं
Output: [EN]नेटवर्क[/EN] ही ऐसा है मैं क्या करूं
EN    : ['नेटवर्क']

Input : जंगल में कैंपिंग और एडवेंचर करने गए
Output: जंगल में [EN]कैंपिंग[/EN] और [EN]एडवेंचर[/EN] करने गए
EN    : ['कैंपिंग', 'एडवेंचर']

Input : पुराने सॉन्ग सुनने में मज़ा आता है
Output: पुराने [EN]सॉन्ग[/EN] सुनने में मज़ा आता है
EN    : ['सॉन्ग']

Input : आपका टॉपिक क्या है मैडम
Output: आपका [EN]टॉपिक[/EN] क्या है [EN]मैडम[/EN]
EN    : ['टॉपिक', 'मैडम']



In [ ]:


def run_pipeline(text):
    """
    Apply the complete cleanup pipeline to one segment.
    Step 1: Number normalization
    Step 2: English word tagging (on the normalized text)
    """
    normalized, num_replacements = normalize_numbers(text)
    tagged, english_words        = tag_english_words(normalized)
    return {
        'original'         : text,
        'after_num_norm'   : normalized,
        'final_tagged'     : tagged,
        'num_replacements' : num_replacements,
        'english_words'    : english_words,
    }


all_results = []
for t in transcriptions:
    if not t.get('raw'):
        continue
    for seg in t['raw']:
        if not isinstance(seg, dict):
            continue
        seg_text = seg.get('text', '').strip()
        if len(seg_text) < 10:
            continue
        result = run_pipeline(seg_text)
        result['recording_id'] = t['recording_id']
        result['start']        = seg.get('start', 0)
        all_results.append(result)

print(f'Pipeline complete.')
print(f'  Total segments processed : {len(all_results)}')
print(f'  With number conversions  : {sum(1 for r in all_results if r["num_replacements"])}')
print(f'  With English words       : {sum(1 for r in all_results if r["english_words"])}')

Pipeline complete.
  Total segments processed : 4137
  With number conversions  : 782
  With English words       : 1350


In [ ]:
# Before / After Examples from Actual Data

print('=' * 70)
print('PART A — NUMBER NORMALISATION: 5 Before/After Examples')
print('=' * 70)
print()

# Show segments that actually had number conversions
num_examples = [r for r in all_results if r['num_replacements']]

if len(num_examples) >= 5:
    for r in num_examples[:5]:
        print(f'BEFORE: {r["original"][:180]}')
        print(f'AFTER : {r["after_num_norm"][:180]}')
        print(f'Conv  : {r["num_replacements"]}')
        print()
else:
    # Supplement with embedded examples from data analysis
    embedded_num = [
        'छः सात आठ किलोमीटर में नौ बजे है',
        'पूरे अस्सी घाट होंगे वहां पे',
        'दो सौ रुपया लेता है पूरा अस्सी घाट',
        'सुबह दस बज गया था',
        'नाइन्टीन नाइन्टीस के सॉन्ग हो गए ना',
    ]
    for ex in embedded_num:
        norm, reps = normalize_numbers(ex)
        print(f'BEFORE: {ex}')
        print(f'AFTER : {norm}')
        print(f'Conv  : {reps}')
        print()
    for r in num_examples:
        print(f'BEFORE: {r["original"][:180]}')
        print(f'AFTER : {r["after_num_norm"][:180]}')
        print(f'Conv  : {r["num_replacements"]}')
        print()

print()
print('=' * 70)
print('PART B — ENGLISH WORD DETECTION: 5 Tagged Examples')
print('=' * 70)
print()

en_examples = [r for r in all_results if len(r['english_words']) >= 2]

if len(en_examples) >= 5:
    for r in en_examples[:5]:
        print(f'INPUT  : {r["after_num_norm"][:200]}')
        print(f'TAGGED : {r["final_tagged"][:220]}')
        print(f'EN     : {r["english_words"]}')
        print()
else:
    embedded_en = [
        'मेरा इंटरव्यू बहुत अच्छा गया और मुझे जॉब मिल गई',
        'ये problem solve नहीं हो रहा है',
        'हमारा प्रोजेक्ट भी था उस एरिया में',
        'बोल्ड आउटफिट नहीं पहनना चाहिए पब्लिकली',
        'नेटवर्क ही ऐसा है इमोशनल हो गया था',
    ]
    for ex in embedded_en:
        tagged, en_words = tag_english_words(ex)
        print(f'INPUT  : {ex}')
        print(f'TAGGED : {tagged}')
        print(f'EN     : {en_words}')
        print()
    for r in en_examples:
        print(f'INPUT  : {r["after_num_norm"][:200]}')
        print(f'TAGGED : {r["final_tagged"][:220]}')
        print(f'EN     : {r["english_words"]}')
        print()

PART A — NUMBER NORMALISATION: 5 Before/After Examples

BEFORE: अब काफी अच्छा होता है क्योंकि उनकी जनसंख्या बहुत कम दी जा रही है तो हमें उनको देखना था तो एक देखना था मतलब वो तो देखना था लेकिन हमारा प्रोजेक्ट भी था कि जो जन जाती पाई जाती है उधर 
AFTER : अब काफी अच्छा होता है क्योंकि उनकी जनसंख्या बहुत कम दी जा रही है तो हमें उनको देखना था तो 1 देखना था मतलब वो तो देखना था लेकिन हमारा प्रोजेक्ट भी था कि जो जन जाती पाई जाती है उधर क
Conv  : [('एक', '1', 90)]

BEFORE: हां तो फिर वहां जो दिन भर तो दिन में तो खोजने में वक्त बीत गया। जब रात की बारी आई तो हमने टेंट गड़ा और रहा तो जब पता जैसी रात हुआ ना शाम मतलब छै सात में इतना
AFTER : हां तो फिर वहां जो दिन भर तो दिन में तो खोजने में वक्त बीत गया। जब रात की बारी आई तो हमने टेंट गड़ा और रहा तो जब पता जैसी रात हुआ ना शाम मतलब 6 7 में इतना
Conv  : [('छै', '6', 142), ('सात', '7', 145)]

BEFORE: ऐसा ही बात था तो पता है रात को मतलब जैसे छः सात आठ किलोमीटर में नौ बजे है नौ उसके बाद लेकिन शांति लेकिन सुकुन बहुत मिला और जैसे सुबह हुआ ना जैसे घर में तो 

In [ ]:
#  Statistics and Analysis

total      = len(all_results)
with_nums  = sum(1 for r in all_results if r['num_replacements'])
with_en    = sum(1 for r in all_results if r['english_words'])
all_en     = [w for r in all_results for w in r['english_words']]
en_counter = Counter(all_en)

print('=== PIPELINE STATISTICS ===')
print(f'Total segments processed    : {total}')
print(f'Segments with number conv.  : {with_nums} ({with_nums/max(total,1)*100:.1f}%)')
print(f'Segments with English words : {with_en} ({with_en/max(total,1)*100:.1f}%)')
print(f'Total EN word occurrences   : {len(all_en)}')
print(f'Unique EN words detected    : {len(en_counter)}')
print()

print('Top 25 detected English words:')
print(f'{"Word":<25} {"Count":>6}  Detection method')
print('-' * 60)
for word, count in en_counter.most_common(25):
    method = 'known list' if word in KNOWN_ENGLISH_DEVANAGARI else 'phonotactic'
    print(f'{word:<25} {count:>6}  {method}')

print()
print('=== WHERE THE PIPELINE HELPS ===')
print('''
Number normalization HELPS when:
  ✓ Specific counts: "दस बजे" → "10 बजे" (time parsing)
  ✓ Prices: "दो सौ रुपया" → "200 रुपया" (entity extraction)
  ✓ Years: "दो हजार बारह" → "2012" (temporal tagging)
  ✓ Measurements: "बारह किलोमीटर" → "12 किलोमीटर"

Number normalization must AVOID:
  ✗ Idioms: "दो-चार बातें" → should stay as-is
  ✗ Idioms: "एक से एक" → should stay as-is
  ✗ Ordinals: "पहला बार" → पहला should not become 1st

English detection HELPS when:
  ✓ Downstream NER: know that प्रोजेक्ट = project (entity type)
  ✓ Translation: flag EN words to skip re-translating them
  ✓ TTS: use English pronunciation for tagged words
  ✓ Script normalization: Roman ↔ Devanagari pairing
''')

=== PIPELINE STATISTICS ===
Total segments processed    : 4137
Segments with number conv.  : 782 (18.9%)
Segments with English words : 1350 (32.6%)
Total EN word occurrences   : 2174
Unique EN words detected    : 453

Top 25 detected English words:
Word                       Count  Detection method
------------------------------------------------------------
टॉपिक                        145  known list
कॉलेज                        127  phonotactic
फोन                           99  known list
जॉब                           86  phonotactic
मैम                           57  known list
हेलो                          54  known list
शेयर                          54  known list
मोबाइल                        38  known list
प्रॉब्लम                      29  phonotactic
सॉन्ग                         27  known list
पॉइंट                         27  phonotactic
कॉल                           27  phonotactic
एरिया                         26  known list
रोटी                          25  phonotactic
कंप

In [ ]:
#  Excel

DARK  = 'FF1B3A6B'; LITE  = 'FFD6E4F7'
GREEN = 'FFD9F0D4'; AMBER = 'FFFFF3CD'; GRAY = 'FFF5F5F5'; WHITE = 'FFFFFFFF'
_thin = Side(style='thin', color='FFB0B0B0')
_brd  = Border(left=_thin, right=_thin, top=_thin, bottom=_thin)

def _H(c, v, bg=DARK, fg='FFFFFFFF', bold=True, sz=10):
    c.value = v
    c.font  = Font(bold=bold, color=fg, size=sz, name='Arial')
    c.fill  = PatternFill('solid', start_color=bg)
    c.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
    c.border = _brd

def _D(c, v, bg=WHITE, bold=False, lft=False):
    c.value = v
    c.font  = Font(bold=bold, name='Arial', size=9)
    c.fill  = PatternFill('solid', start_color=bg)
    c.alignment = Alignment(horizontal='left' if lft else 'center',
                            vertical='center', wrap_text=True)
    c.border = _brd

def _W(ws, col, width):
    ws.column_dimensions[get_column_letter(col)].width = width


wb = Workbook()

#  Pipeline output 
ws1 = wb.active; ws1.title = '1. Pipeline Output'; ws1.freeze_panes = 'A3'
ws1.merge_cells('A1:F1')
_H(ws1['A1'], 'Q2 — ASR Cleanup Pipeline | Number Norm + English Detection', sz=12)
ws1.row_dimensions[1].height = 24
hdrs   = ['Rec ID', 'Original', 'After Num Norm', 'Final Tagged', 'Numbers Converted', 'English Words']
widths = [10, 45, 45, 45, 25, 30]
for ci, (h, w) in enumerate(zip(hdrs, widths), 1):
    _H(ws1.cell(2, ci), h); _W(ws1, ci, w)
ws1.row_dimensions[2].height = 24

for ri, r in enumerate(all_results[:600], 3):
    has_change = bool(r['num_replacements'] or r['english_words'])
    bg = AMBER if has_change else WHITE
    _D(ws1.cell(ri,1), r['recording_id'],  bg=GRAY)
    _D(ws1.cell(ri,2), r['original'][:200],      bg=WHITE, lft=True)
    _D(ws1.cell(ri,3), r['after_num_norm'][:200], bg=WHITE, lft=True)
    _D(ws1.cell(ri,4), r['final_tagged'][:200],   bg=WHITE, lft=True)
    num_str = '; '.join(f'"{a}"→{b}' for a, b, _ in r['num_replacements'])
    _D(ws1.cell(ri,5), num_str, bg=GREEN if r['num_replacements'] else WHITE, lft=True)
    _D(ws1.cell(ri,6), ', '.join(r['english_words']),
       bg=AMBER if r['english_words'] else WHITE, lft=True)
    ws1.row_dimensions[ri].height = 14

# Number edge cases 
ws2 = wb.create_sheet('2. Num Norm Edge Cases')
ws2.merge_cells('A1:D1')
_H(ws2['A1'], 'Number Normalisation — Edge Cases and Judgment Calls', sz=12)
ws2.row_dimensions[1].height = 24
for ci, (h, w) in enumerate(zip(['Input', 'Output', 'Decision', 'Reasoning'], [28, 28, 20, 55]), 1):
    _H(ws2.cell(2, ci), h); _W(ws2, ci, w)
ws2.row_dimensions[2].height = 22

for ri, (phrase, decision, reasoning) in enumerate(edge_cases, 3):
    norm, reps = normalize_numbers(phrase)
    _D(ws2.cell(ri,1), phrase,   lft=True)
    _D(ws2.cell(ri,2), norm,     bg=GREEN if 'CONVERT' in decision else AMBER, lft=True)
    _D(ws2.cell(ri,3), decision, bg=GREEN if 'CONVERT' in decision else AMBER, lft=True)
    _D(ws2.cell(ri,4), reasoning, bg=GRAY, lft=True)
    ws2.row_dimensions[ri].height = 42

#  English word frequency 
ws3 = wb.create_sheet('3. English Words')
ws3.merge_cells('A1:C1')
_H(ws3['A1'], 'English Words Detected in Dataset', sz=12)
ws3.row_dimensions[1].height = 24
for ci, (h, w) in enumerate(zip(['Word (Devanagari)', 'Count', 'Method'], [28, 8, 20]), 1):
    _H(ws3.cell(2, ci), h); _W(ws3, ci, w)
ws3.row_dimensions[2].height = 22

for ri, (word, count) in enumerate(en_counter.most_common(150), 3):
    method = 'known list' if word in KNOWN_ENGLISH_DEVANAGARI else 'phonotactic'
    _D(ws3.cell(ri,1), word,   lft=True)
    _D(ws3.cell(ri,2), count,  bg=AMBER if count > 5 else GRAY)
    _D(ws3.cell(ri,3), method, bg=GREEN if method == 'known list' else LITE, lft=True)
    ws3.row_dimensions[ri].height = 14

wb.save(OUTPUT_FILE)
print(f'Saved → {OUTPUT_FILE}')
print('Sheets:', wb.sheetnames)

Saved → Q2_Cleanup_Results.xlsx
Sheets: ['1. Pipeline Output', '2. Num Norm Edge Cases', '3. English Words']


---
## Summary

### Part a — Number Normalization

| Approach | Detail |
|---|---|
| Algorithm | Recursive left-to-right token scan |
| Chaining rule | Only chain tokens when a LARGE multiplier (सौ/हजार/लाख) is present |
| Ordinals | Preserved (पहला, दूसरा) — not converted |
| Idioms | Preserved (दो-चार, एक से एक) — pattern matched |
| Coverage | Units (1–9), teens (10–19), tens (20–90), all irregulars (21–99), hundreds, thousands, lakhs |

**Key design decision:** `दो तीन लोग` → `2 3 लोग` (not `5 लोग`).  
Adjacent plain numbers without a multiplier are converted independently, not summed.

### Part b — English Word Detection

| Layer | Trigger | Example |
|---|---|---|
| 1 | Roman script | `feedback` → `[EN]feedback[/EN]` |
| 2 | Known loanword list (100+ words) | `प्रोजेक्ट` → `[EN]प्रोजेक्ट[/EN]` |
| 3 | Phonotactic pattern | Words ending in -इंग, -शन, ऑ vowel → tagged |

**Important:** Tagged words are **not errors**. They are correctly transcribed English words  
in Devanagari script, flagged so downstream tools can handle them appropriately.